<a href="https://colab.research.google.com/github/donlap/stat424/blob/main/10_RecSys.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Item Recommendation using Bias and Latent Factor Models

In this notebook, we will build Recommender Systems using the Google Restaurants dataset from [here](https://mcauleylab.ucsd.edu/public_datasets/gdrive/googlelocal_restaurants/filter_all_t.json). Specifically, it is a dataset of restaurants reviews from Google Maps posted by users, as well as other metadata for each restaurant.

Here's an example of a single review:
```
{
  "business_id": "605730f68cd0e3d69a52284b",
  "user_id": "113890892872599852766",
  "rating": 4,
  "review_text": "We came for a birthday brunch and this place is so much bigger than it looks from the outside! It was totally packed and loud. Service was on the slower side. I ordered 2 mojitos: 1 lime and 1 mango. The ingredient weren\u2019t really fresh, there wasn\u2019t even any visible mango in the second one. Tasted like mango juice from concentrate. My food was really good though. I ordered the steak and eggs and I usually order my steak rare but this was skirt steak so they did a perfect medium. The sunny side up eggs were more cooked than I would have preferred but still good. They actually cooked the breakfast potatoes too, which was awesome. Will likely be back to try something else!",
  "pics":
    [
      {
        "id": "AF1QipPrls2G30PS3tyC55KBxUrKgy3ER0AB5UJY57BZ",
        "url": ["https://lh5.googleusercontent.com/p/AF1QipPrls2G30PS3tyC55KBxUrKgy3E
R0AB5UJY57BZ=w150-h150-k-no-p"]
      },
      {
        "id": "AF1QipOBdiu4hEPC4538Q9iRzl1gAyNuMxThTVRKubSX",
        "url": ["https://lh5.googleusercontent.com/p/AF1QipOBdiu4hEPC4538Q9iRzl1gAyNu
MxThTVRKubSX=w150-h150-k-no-p"]
      }
    ]
}
```

**Reference.**
Personalized Showcases: Generating Multi-Modal Explanations for Recommendations
An Yan, Zhankui He, Jiacheng Li, Tianyang Zhang, Julian Mcauley
arXiv:2207.00422, 2022.


Using this dataset, we will train two models, namely the **bias model** and the **latent factor model** using SGD and ALS algorithms.

## 1. Data Loading & Matrix Setup


In [1]:
import json
import pandas as pd
import numpy as np
import jax
import jax.numpy as jnp
import optax


!wget -q -O filter_all_t.json https://mcauleylab.ucsd.edu/public_datasets/gdrive/googlelocal_restaurants/filter_all_t.json

with open('filter_all_t.json', 'r') as f:
    data = json.load(f)

# Extract the training set
train_data = data['train'] if isinstance(data, dict) and 'train' in data else data
df_raw = pd.DataFrame(train_data)

df_raw = df_raw.head(5000)



Loaded 5000 ratings for 4545 users and 4137 restaurants. Global Mean: 4.46


Let's take a look at the dataset.

In [15]:
df_raw

,business_id,user_id,rating,review_text,pics,history_reviews
0,60567465d335d0abfb415b26,101074926318992653684,4,The tang of the tomato sauce is outstanding. A...,"[AF1QipM-2IRmvitARbcJr7deWfe5hyVBg_ArPMQSYvq0,...",[[101074926318992653684_6056272797d555cc6fb0d1...
1,6050fa9f5b4ccec8d5cae994,117065749986299237881,5,Chicken and waffles were really good!,[AF1QipMpfxIZUT_aymQ3qPGO-QgGYzxbtLZGmHufAp2s],[[117065749986299237881_605206f8d8c08f462b93e8...
2,604be10877e81aaed3cc9a1e,106700937793048450809,4,The appetizer of colossal shrimp was very good...,"[AF1QipMNnqM5X9sSyZ9pXRZ1jvrURHN9bZhGdzuEXoP8,...",[[106700937793048450809_6044300b27f39b7b5d1dbf...
3,60411e017cd8bf130362365a,101643045857250355161,5,The fish tacos here omg! The salad was great ...,"[AF1QipM-a6AGGp4Hgk5RD0gY5sDRp5kEfB1hZLvlRkft,...",[[101643045857250355161_604fbdd099686c10168c91...
4,604139dd7cd8bf1303624208,109802745326785766951,4,"Ribs are great, as are the mac and cheese, fri...",[AF1QipNVys4yq-5w_3EsDdHpSc9ZNb7Nl30Mfb6Y0Gup],[[109802745326785766951_60524fa9f09a4ffff042f9...
...,...,...,...,...,...,...
4995,6056ffbff69c7b117807028e,114675567611442677958,4,Good selection of rolls and sushi.,"[AF1QipNy2fo62caTv7IVAIywTBy3G05Lyby0R7ydPByF,...",[[114675567611442677958_6051ac77da79151bfc125a...
4996,6046b9b8b0e2129e47535591,117535197913796753690,4,that spicy calamari is a must-try! Beef burrit...,"[AF1QipN8B_4Djqyo6FhLylMrT9JFvldWyI9FyOGRN1Wr,...",[[117535197913796753690_604dec44b9a3d5528c50bb...
4997,60415badc6fcf1fddba12f1f,100330246459789646772,5,We shared a Salumeria tasting with their Neapo...,"[AF1QipPLiv_v3la19RlflXY5uVq6lyZAzAdrPIK6hS3v,...",[[100330246459789646772_60415b042e57ebdea29c3a...
4998,604ca35a5a9e6adec8bf8cb1,116906648024455485412,4,"Know that their pancakes are huge, and you hav...",[AF1QipNYRVTCFRH-2OoQwfOwWnKSDwe3PYqlxLL8Spc],[[116906648024455485412_60497945b1a0aaee3eefae...


We extract from this dataset vectors of users IDs, items' IDs, and the matrix of ratings. We will also create a binary matrix `M` that indicates whether each user has given a rating to each movie.

In [ ]:
# Map String IDs to Continuous Integers (0 to N-1)
df = pd.DataFrame()
# pd.factorize assigns a unique integer to each unique string it finds
df['user'], user_uniques = pd.factorize(df_raw['user_id'])
df['item'], item_uniques = pd.factorize(df_raw['business_id'])
df['rating'] = df_raw['rating'].astype(float)

num_users = df['user'].nunique()
num_items = df['item'].nunique()
global_mean = df['rating'].mean()

print(f"Loaded {len(df)} ratings for {num_users} users and {num_items} restaurants. Global Mean: {global_mean:.2f}")

# Create Dense Matrices (Required for ALS)
R = np.zeros((num_users, num_items))
M = np.zeros((num_users, num_items))

for _, row in df.iterrows():
    u, i, r = int(row['user']), int(row['item']), row['rating']
    R[u, i] = r
    M[u, i] = 1.0  # Mask matrix indicating a rating exists

# Convert to JAX arrays for downstream processing
R_matrix, M_matrix = jnp.array(R), jnp.array(M)
u_arr, i_arr, r_arr = df['user'].values, df['item'].values, df['rating'].values

In [17]:
R_matrix

Array([[4., 0., 0., ..., 0., 0., 0.],
       [0., 5., 0., ..., 0., 0., 0.],
       [0., 0., 4., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 5., 0., 0.],
       [0., 0., 0., ..., 0., 0., 4.]], dtype=float32)

## 2. Baseline Bias Model (SGD)
**Formula:** $Rating \approx \alpha + \beta_u + \beta_i$

We predict a rating by taking the global average ($\alpha$), and adding a user-specific bias ($\beta_u$) and an item-specific bias ($\beta_i$).

**Objective Loss Function:**
$$ L = \sum (r_{ui} - (\alpha + \beta_u + \beta_i))^2 + \lambda (\beta_u^2 + \beta_i^2) $$

**SGD Update Equations:**
Let the error be $e_{ui} = r_{ui} - \hat{r}_{ui}$. Given a learning rate $\eta$ and regularization $\lambda$:
* $\alpha \leftarrow \alpha + \eta \cdot e_{ui}$
* $\beta_u \leftarrow \beta_u + \eta \cdot (e_{ui} - \lambda \beta_u)$
* $\beta_i \leftarrow \beta_i + \eta \cdot (e_{ui} - \lambda \beta_i)$

First, we set up parameters. We will use Adam as the optimizer.

In [7]:
base_params_sgd = {'a': global_mean, 'bu': jnp.zeros(num_users), 'bi': jnp.zeros(num_items)}

LAMBDA = 0.01
optimizer = optax.adam(learning_rate=0.1)
opt_state = optimizer.init(base_params_sgd)

Epoch 0 | Loss: 0.7140
Epoch 10 | Loss: 0.8273
Epoch 20 | Loss: 0.7441
Epoch 30 | Loss: 0.7003
Epoch 40 | Loss: 0.6901
Epoch 50 | Loss: 0.6878
Epoch 60 | Loss: 0.6869
Epoch 70 | Loss: 0.6864
Epoch 80 | Loss: 0.6861
Epoch 90 | Loss: 0.6861


We then define the loss function $ L = \sum (r_{ui} - (\alpha + \beta_u + \beta_i))^2 + \lambda (\beta_u^2 + \beta_i^2) $.

In [ ]:
def base_predict(params, u, i):
    return params['a'] + params['bu'][u] + params['bi'][i]

@jax.jit
def base_loss(params, u, i, r):
    preds = base_predict(params, u, i)
    mse = jnp.mean((r - preds)**2)
    reg = LAMBDA * (jnp.sum(params['bu']**2) + jnp.sum(params['bi']**2))
    return mse + reg

Finally, we define one SGD iteration. Using this, we train the model for 100 iterations.

In [ ]:
@jax.jit
def base_sgd_step(params, opt_state, u, i, r):
    loss, grads = jax.value_and_grad(base_loss)(params, u, i, r)
    updates, opt_state = optimizer.update(grads, opt_state)
    return optax.apply_updates(params, updates), opt_state, loss


for epoch in range(100):
    base_params_sgd, opt_state, loss = base_sgd_step(base_params_sgd, opt_state, u_arr, i_arr, r_arr)
    if epoch % 10 == 0: print(f"Epoch {epoch} | Loss: {loss:.4f}")

With the trained parameters, let's recommend some items for User #0.

In [14]:
uid = 0

preds = base_params_sgd['a'] + base_params_sgd['bu'][uid] + base_params_sgd['bi']
preds = preds.at[i_arr[u_arr == 0]].set(-jnp.inf)
top_10_items = jnp.argsort(preds)[-10:][::-1]

top_10_items

Array([ 310,   98,  477,  288,  206,  962,   94, 1225, 1370,  298], dtype=int32)

## 3. Baseline Bias Model (ALS)

Instead of gradient descent, Alternating Least Squares (ALS) freezes one parameter and solves for the exact mathematical minimum of the other.

**ALS Update Equations:**
Fixing $\alpha$ and $\beta_i$, we solve for $\beta_u$:
$$ \beta_u = \frac{\sum_{i \in I_u} (r_{ui} - \alpha - \beta_i)}{|I_u| + \lambda} $$

Fixing $\alpha$ and $\beta_u$, we solve for $\beta_i$:
$$ \beta_i = \frac{\sum_{u \in U_i} (r_{ui} - \alpha - \beta_u)}{|U_i| + \lambda} $$

We again initialize the parameters. However, unlike SGD, we don't need an optimizer for ALS.

In [28]:
base_params_als = {'a': global_mean, 'bu': jnp.zeros(num_users), 'bi': jnp.zeros(num_items)}

LAMBDA = 0.01

Then, we define a single ALS iteration.

In [29]:
@jax.jit
def base_als_step(R, M, params):
    a = params['a']
    # Solve for User Bias (fixing item bias)
    bu = jnp.sum(M * (R - a - params['bi']), axis=1) / (jnp.sum(M, axis=1) + LAMBDA)

    # Solve for Item Bias (fixing user bias)
    bi = jnp.sum(M * (R - a - bu[:, None]), axis=0) / (jnp.sum(M, axis=0) + LAMBDA)

    return {'a': a, 'bu': bu, 'bi': bi}

Finally, we train the model for 20 iterations.

In [30]:
for epoch in range(20):
    base_params_als = base_als_step(R_matrix, M_matrix, base_params_als)

    preds = base_params_als['a'] + base_params_als['bu'][:, None] + base_params_als['bi'][None, :]
    loss = jnp.sum(M_matrix * (R_matrix - preds)**2) / jnp.sum(M_matrix)
    print(f"Epoch {epoch} | MSE: {loss:.6f}")

Epoch 0 | MSE: 0.006770
Epoch 1 | MSE: 0.001270
Epoch 2 | MSE: 0.000497
Epoch 3 | MSE: 0.000285
Epoch 4 | MSE: 0.000191
Epoch 5 | MSE: 0.000137
Epoch 6 | MSE: 0.000104
Epoch 7 | MSE: 0.000081
Epoch 8 | MSE: 0.000066
Epoch 9 | MSE: 0.000055
Epoch 10 | MSE: 0.000047
Epoch 11 | MSE: 0.000041
Epoch 12 | MSE: 0.000037
Epoch 13 | MSE: 0.000034
Epoch 14 | MSE: 0.000031
Epoch 15 | MSE: 0.000029
Epoch 16 | MSE: 0.000028
Epoch 17 | MSE: 0.000027
Epoch 18 | MSE: 0.000026
Epoch 19 | MSE: 0.000026


Let's recommend 10 items to User #0

In [31]:
uid = 0

preds = base_params_als['a'] + base_params_als['bu'][uid] + base_params_als['bi']
preds = preds.at[i_arr[u_arr == 0]].set(-jnp.inf)
top_10_items = jnp.argsort(preds)[-10:][::-1]

top_10_items

Array([1164, 2279,   15,  575, 3593, 3448, 3812, 2443,  578, 1669], dtype=int32)

## 4. Latent Factor Model (SGD)
**Formula:** $Rating \approx \alpha + \beta_u + \beta_i + \gamma_u \cdot \gamma_i$

We extend the baseline by adding a dot product between a user vector $\gamma_u$ and an item vector $\gamma_i$. This captures complex interactions (e.g., this user loves this specific genre).

**SGD Update Equations:**
Let $e_{ui} = r_{ui} - \hat{r}_{ui}$.
* $\alpha, \beta_u, \beta_i$ updates remain the same.
* $\gamma_u \leftarrow \gamma_u + \eta \cdot (e_{ui} \gamma_i - \lambda \gamma_u)$
* $\gamma_i \leftarrow \gamma_i + \eta \cdot (e_{ui} \gamma_u - \lambda \gamma_i)$

We first initialize the parameters and the optimizer. For the users' and items' latent variables, we randomly initialize the values from the normal distribution with scale = 0.1.

In [45]:
key = jax.random.PRNGKey(42)
k1, k2 = jax.random.split(key)
K = 5 # Dimension of Latent Variables

lf_params_sgd = {
    'a': global_mean, 'bu': jnp.zeros(num_users), 'bi': jnp.zeros(num_items),
    'pu': jax.random.normal(k1, (num_users, K)) * 0.1,
    'qi': jax.random.normal(k2, (num_items, K)) * 0.1
}

LAMBDA = 0.05
optimizer = optax.adam(learning_rate=0.1)
opt_state_lf = optimizer.init(lf_params_sgd)

Here's the loss function for the latent factor model:

In [46]:
def lf_predict(params, u, i):
    dot = jnp.sum(params['pu'][u] * params['qi'][i], axis=-1)
    return params['a'] + params['bu'][u] + params['bi'][i] + dot

@jax.jit
def lf_loss(params, u, i, r):
    preds = lf_predict(params, u, i)
    mse = jnp.mean((r - preds)**2)
    reg = LAMBDA * (jnp.sum(params['bu']**2) + jnp.sum(params['bi']**2) +
                    jnp.sum(params['pu']**2) + jnp.sum(params['qi']**2))
    return mse + reg

Finally, we define a single SGD iteration. Using this function, we train the model for 100 iterations.

In [47]:
@jax.jit
def lf_sgd_step(params, opt_state, u, i, r):
    loss, grads = jax.value_and_grad(lf_loss)(params, u, i, r)
    updates, opt_state = optimizer.update(grads, opt_state)
    return optax.apply_updates(params, updates), opt_state, loss


for epoch in range(100):
    lf_params_sgd, opt_state_lf, loss = lf_sgd_step(lf_params_sgd, opt_state_lf, u_arr, i_arr, r_arr)
    if epoch % 10 == 0: print(f"Epoch {epoch} | Loss: {loss:.4f}")

Epoch 0 | Loss: 22.6058
Epoch 10 | Loss: 4.6758
Epoch 20 | Loss: 1.9903
Epoch 30 | Loss: 1.0530
Epoch 40 | Loss: 0.8480
Epoch 50 | Loss: 0.7602
Epoch 60 | Loss: 0.7279
Epoch 70 | Loss: 0.7148
Epoch 80 | Loss: 0.7108
Epoch 90 | Loss: 0.7094


Using this model, we recommend 10 items to User #0.

In [52]:
preds_lf = (lf_params_sgd['a'] + lf_params_sgd['bu'][0] + lf_params_sgd['bi']
                               + (lf_params_sgd['qi'] @ lf_params_sgd['pu'][0]))
preds_lf = preds_lf.at[i_arr[u_arr == 0]].set(-jnp.inf)
top_10_items_lf = jnp.argsort(preds_lf)[-10:][::-1]

top_10_items_lf

Array([  98,  310,  477,  206,  288, 1370, 1225,   94,  962,  298], dtype=int32)

## 5. Latent Factor Model (ALS)

**ALS Update Equations:**
We can elegantly solve for the bias and the latent factors simultaneously using Ridge Regression.
Let $x_u =[\beta_u, \gamma_u]^T$ and $Y_i = [1, \gamma_i]^T$.

Fixing Items, update User $u$:
$$ x_u = \left( \sum_{i \in I_u} Y_i Y_i^T + \lambda I \right)^{-1} \sum_{i \in I_u} (r_{ui} - \alpha - \beta_i) Y_i $$

Fixing Users, update Item $i$:
$$ Y_i = \left( \sum_{u \in U_i} x_u x_u^T + \lambda I \right)^{-1} \sum_{u \in U_i} (r_{ui} - \alpha - \beta_u) x_u $$

For the ALS algorithm, we only need to initialize the parameters.

In [87]:
lf_params_als = {
    'a': global_mean, 'bu': jnp.zeros(num_users), 'bi': jnp.zeros(num_items),
    'pu': jax.random.normal(k1, (num_users, K)) * 0.1,
    'qi': jax.random.normal(k2, (num_items, K)) * 0.1
}

LAMBDA = 0.001

The update equation
$$ x_u = \left( \sum_{i \in I_u} Y_i Y_i^T + \lambda I \right)^{-1} \sum_{i \in I_u} (r_{ui} - \alpha - \beta_i) Y_i $$
is the same as solving the following OLS regression for $x_u$:
$$A_u x_u = b_u, $$
where $A_u= \sum_{i \in I_u} Y_i Y_i^T + \lambda I$ and $b_u = \sum_{i \in I_u} (r_{ui} - \alpha - \beta_i) Y_i$.

Similarly, the update equation
$$ Y_i = \left( \sum_{u \in U_i} x_u x_u^T + \lambda I \right)^{-1} \sum_{u \in U_i} (r_{ui} - \alpha - \beta_u) x_u $$
is the same as solving the following OLS regression for $Y_i$:
$$ A_i Y_i =  b_i,  $$
where $A_i = \sum_{u \in U_i} x_u x_u^T + \lambda I $ and $b_i = \sum_{u \in U_i} (r_{ui} - \alpha - \beta_u) x_u$.

Therefore, we define the following utility function for OLS regression.

In [88]:
@jax.jit
def solve_ridge(A, b, lam):
    return jnp.linalg.solve(A + lam * jnp.eye(A.shape[0]), b)

Using this function, we define a single ALS iteration for the latent factor model.

In [89]:
@jax.jit
def lf_als_step(R, M, params):
    a = params['a']

    # 1. Update Users (bu and pu simultaneously)
    Y = jnp.hstack([jnp.ones((num_items, 1)), params['qi']])
    def update_user(u):
        Y_masked = M[u, :, None] * Y
        A_u = Y_masked.T @ Y
        b_u = Y_masked.T @ (R[u, :] - a - params['bi'])
        return solve_ridge(A_u, b_u, LAMBDA)

    X_new = jax.vmap(update_user)(jnp.arange(num_users))
    new_bu, new_pu = X_new[:, 0], X_new[:, 1:]

    # 2. Update Items (bi and qi simultaneously)
    X = jnp.hstack([jnp.ones((num_users, 1)), new_pu])
    def update_item(i):
        X_masked = M[:, i, None] * X
        A_i = X_masked.T @ X
        b_i = X_masked.T @ (R[:, i] - a - new_bu)
        return solve_ridge(A_i, b_i, LAMBDA)

    Y_new = jax.vmap(update_item)(jnp.arange(num_items))
    new_bi, new_qi = Y_new[:, 0], Y_new[:, 1:]

    return {'a': a, 'bu': new_bu, 'bi': new_bi, 'pu': new_pu, 'qi': new_qi}

Finally, we train the model for 10 iterations.

In [90]:
for epoch in range(60):
    lf_params_als = lf_als_step(R_matrix, M_matrix, lf_params_als)

    preds = lf_params_als['a'] + lf_params_als['bu'][:, None] + lf_params_als['bi'][None, :] + lf_params_als['pu'] @ lf_params_als['qi'].T
    loss = jnp.sum(M_matrix * (R_matrix - preds)**2) / jnp.sum(M_matrix)
    print(f"Epoch {epoch} | MSE: {loss:.10f}")

Epoch 0 | MSE: 0.0000001240
Epoch 1 | MSE: 0.0000001106
Epoch 2 | MSE: 0.0000001019
Epoch 3 | MSE: 0.0000000958
Epoch 4 | MSE: 0.0000000911
Epoch 5 | MSE: 0.0000000873
Epoch 6 | MSE: 0.0000000840
Epoch 7 | MSE: 0.0000000813
Epoch 8 | MSE: 0.0000000790
Epoch 9 | MSE: 0.0000000769
Epoch 10 | MSE: 0.0000000749
Epoch 11 | MSE: 0.0000000731
Epoch 12 | MSE: 0.0000000713
Epoch 13 | MSE: 0.0000000696
Epoch 14 | MSE: 0.0000000681
Epoch 15 | MSE: 0.0000000667
Epoch 16 | MSE: 0.0000000654
Epoch 17 | MSE: 0.0000000640
Epoch 18 | MSE: 0.0000000627
Epoch 19 | MSE: 0.0000000613
Epoch 20 | MSE: 0.0000000598
Epoch 21 | MSE: 0.0000000583
Epoch 22 | MSE: 0.0000000568
Epoch 23 | MSE: 0.0000000553
Epoch 24 | MSE: 0.0000000540
Epoch 25 | MSE: 0.0000000528
Epoch 26 | MSE: 0.0000000517
Epoch 27 | MSE: 0.0000000506
Epoch 28 | MSE: 0.0000000496
Epoch 29 | MSE: 0.0000000487
Epoch 30 | MSE: 0.0000000478
Epoch 31 | MSE: 0.0000000470
Epoch 32 | MSE: 0.0000000463
Epoch 33 | MSE: 0.0000000457
Epoch 34 | MSE: 0.000000

Let's recommend 10 items to User #0 using this model.

In [91]:
preds_lf = (lf_params_als['a'] + lf_params_als['bu'][0] + lf_params_als['bi']
                               + (lf_params_als['qi'] @ lf_params_als['pu'][0]))
preds_lf = preds_lf.at[i_arr[u_arr == 0]].set(-jnp.inf)
top_10_items_lf = jnp.argsort(preds_lf)[-10:][::-1]

top_10_items_lf

Array([2151, 3216, 2996, 3960, 3811, 2774,  531, 3689,  291, 4068], dtype=int32)

## 6. Model Evaluation
Finally, we measure how well the algorithms capture user preferences by evaluating **Precision@K**, **Recall@K**, and **MRR (Mean Reciprocal Rank)**.

### 1. Precision@K

Precision measures the proportion of recommended items in the top-K set that are actually relevant.

$$\text{Precision@K} = \frac{|\text{Relevant Items} \cap \text{Top-K Recommended Items}|}{K}$$

### 2. Recall@K

Recall measures the proportion of all relevant items that the model successfully managed to include in the top-K recommendations.

$$\text{Recall@K} = \frac{|\text{Relevant Items} \cap \text{Top-K Recommended Items}|}{|\text{Relevant Items}|}$$

### 3. Mean Reciprocal Rank (MRR)

MRR looks at where the *first* relevant item appeared in the recommendation list. It calculates the reciprocal of that rank ($\text{rank}_u$) for each user, and then averages it across all users ($|U|$).

$$\text{MRR} = \frac{1}{|U|} \sum_{u=1}^{|U|} \frac{1}{\text{rank}_{u}}$$

*(Note: If the model fails to recommend any relevant items for a user, that user's reciprocal rank is $0$).*

In [92]:
def evaluate_metrics(params, predict_fn, K=10):
    # To measure accuracy, we define a relevant movie as one the user rated 4 or 5.
    liked_items = df[df['rating'] >= 4].groupby('user')['item'].apply(set).to_dict()

    precisions, recalls, mrrs = [], [],[]
    all_items = jnp.arange(num_items)

    for u, true_items in liked_items.items():
        if len(true_items) == 0: continue

        user_array = jnp.full(num_items, u)
        preds = predict_fn(params, user_array, all_items)

        # Top K predictions
        top_indices = np.argsort(np.array(preds))[::-1]
        top_k = set(top_indices[:K])

        hits = len(top_k.intersection(true_items))
        precisions.append(hits / K)
        recalls.append(hits / len(true_items))

        # MRR
        mrr = 0.0
        for rank, item in enumerate(top_indices):
            if item in true_items:
                mrr = 1.0 / (rank + 1)
                break
        mrrs.append(mrr)

    return np.mean(precisions), np.mean(recalls), np.mean(mrrs)

evals =[
    ('Baseline (SGD)', base_params_sgd, base_predict),
    ('Baseline (ALS)', base_params_als, base_predict),
    ('Latent Factor (SGD)', lf_params_sgd, lf_predict),
    ('Latent Factor (ALS)', lf_params_als, lf_predict)
]

results =[]
for name, params, predict_fn in evals:
    p, r, m = evaluate_metrics(params, predict_fn, K=10)
    results.append((name, p, r, m))

results_df = pd.DataFrame(results, columns=['Model', 'Precision@10', 'Recall@10', 'MRR'])
display(results_df)

,Model,Precision@10,Recall@10,MRR
0,Baseline (SGD),0.001734,0.016072,0.008613
1,Baseline (ALS),0.000248,0.002311,0.002428
2,Latent Factor (SGD),0.001734,0.016072,0.009730
3,Latent Factor (ALS),0.002798,0.018615,0.014770


According to all three scores, the latent factor models trained with the ALS algorithm provides better recommendations than the other three combinations of model/algorithm.